In [1]:
pip install jellyfish

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 KB 1.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [9]:
import jellyfish
from statistics import mode
import numpy as np
#jellyfish.levenshtein_distance('jellyfish', 'smellyfish')

2

In [2]:
def postprocess_text(preds, labels):
        preds = [pred.strip() for pred in preds]
        labels = [label.strip() for label in labels]

        return preds, labels

In [11]:
def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        if data_args.ignore_pad_token_for_loss:
            # Replace -100 in the labels as we can't decode them.
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # Some simple post-processing
        decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

        delimiter = ", "
        
        # define example evaluation
        def evaluate_example(predict_str: str, ground_str: str):
            predict_spans = predict_str.split(delimiter)
            ground_spans = ground_str.split(delimiter)
            predict_values = defaultdict(lambda: 0)
            ground_values = defaultdict(lambda: 0)
            for span in predict_spans:
                try:
                    predict_values[float(span)] += 1
                except ValueError:
                    predict_values[span.strip()] += 1
            for span in ground_spans:
                try:
                    ground_values[float(span)] += 1
                except ValueError:
                    ground_values[span.strip()] += 1
            _is_correct = predict_values == ground_values
            return _is_correct

        def get_denotation_accuracy_and_levenshtein_dist(predictions: List[str], references: List[str]):
            assert len(predictions) == len(references)
            correct_num = 0
            lev_dist = []
            for predict_str, ground_str in zip(predictions, references):
                lev_dist.append(jellyfish.levenshtein_distance(predict_str.lower().strip(), ground_str.lower().strip()))
                is_correct = evaluate_example(predict_str.lower(), ground_str.lower())
                if is_correct:
                    correct_num += 1
            return correct_num / len(predictions), lev_dist.mean(), lev_dist.std(),mode(lev_dist)
        
        accuracy, mean, std, l_mode = get_denotation_accuracy_and_levenshtein_dist(decoded_preds, decoded_labels)
        result = {"denotation_accuracy": accuracy,
                 "levenshtein_dist_mean": mean,
                 "levenshtein_dist_std":std,
                "levenshtein_dist_mode":l_mode
                 }

        return result

In [10]:
mode([1,2,2,5,6,7])

2